# Datamine PANELK Process Example

This notebook demonstrates how to configure and run the **`panelk`** process wrapper in `dmstudio`.

## Process Description

## PANELK

See also [PANELEST Process](<panelest.md>).

This process estimates the average value and the estimation variance of irregular shaped 2-D panels using kriging.

Panels are defined as a series of 2-D points in a standard string file. The value of **ZP** is ignored. Several panels may be held in the same file, all of which will be evaluated in a single run of the process. It does not matter whether the panel strings are open or closed. **VMETHOD** controls the method by which the variogram model is defined:

## VMETHOD=1

A single structure spherical model is used, defined by parameters **NUGGET** and so on. This method is consistent with earlier versions of the process, and is the default.

## VMETHOD=2

The standard range of variogram models is available, defined by interactive prompting. This is consistent with processes **KRG3DB** , **PTK3DA** and so on. **VMETHOD** =1 then the two variogram ranges are defined by **RANGE1** and **RANGE2** , and the anisotropy angle by **VGANGLE1**.

* **Note** (The angle is measured clockwise from the North to direction 1. Direction 2 is perpendicular to direction 1 in the same plane as the panels. It does not matter whether direction 1 represents the major or minor axis.):

The following definitions are equivalent:

## * RANGE1 100 300 100

## * RANGE2 300 100 300

## * VGANGLE1 50 140 230

If **VMETHOD** =2 then the user is prompted interactively to enter the variogram parameters for all 3 directions. The data for the 3rd direction, perpendicular to the plane of the panels, is not used, but values must still be entered for that direction. It is suggested that values in the third direction are set equal to the values in direction 1 or direction 2. Whatever values are entered for the third direction,the range must not be set to zero. - only samples within a distance **SEARCH** of the centre of gravity (CG) ofthe panel are selected. - if more than **NMAX** lie within the search area, then the nearest **NMAX** to the CG are selected.

## **NSIM**

The origin of the grid is at a point defined by the minimum X and Y of the panel points. **MINSIM** then the value of **NSIM** is increased by 1, and the process is repeated until the number of internal points exceeds **MINSIM**. The maximum number of simulated points allowed is 1000. Processing speed is approximately proportional to the square of the number of samples in the kriging matrix. It is also influenced to a lesser extent by the number of simulated points in the panel.

* **Warning** (If two or more samples have the same X and Y coordinate then an error will be reported as _ERR 131 in SOLVED_ and _ERR 122 in PKM3DA_.):

### Input Files:

* **in1** (Undefined):
  Input sample data file. This must contain the three fields **X , Y** and **VALUE**.
  Required=Yes

* **in2** (Undefined):
  The input panel data file. This must contain the 4 fields **PANEL , PTN , XP** field
  must be defined explicitly, and will often be called **PVALUE**. The strings may be open
  or closed.
  Required=Yes

### Output Files:

* **out** (Undefined):
  The output results file. This will contain the following fields:
  Required=No

### Fields:

* **x** (Numeric : IN):
  X coordinate of sample data in the IN file.
  Default=X
  Required=Yes

* **y** (Numeric : IN):
  Y coordinate of sample data in the IN file.
  Default=Y
  Required=Yes

* **value** (Numeric : IN):
  Field to be kriged in the **IN** file.
  Default=Undefined
  Required=Yes

* **panel** (Numeric : IN2):
  Panel identifier. This is a numeric field in the **IN2** file. Its name will often be

## **PVALUE**.

  Default=PVALUE
  Required=Yes

### Parameters:

* **nugget**:
  Nugget variance (0).
  Range=Undefined
  Values=Undefined
  Default=0
  Required=No

* **var1**:
  Spatial variance (1). The difference between the nugget variance
  Range=Undefined
  Values=Undefined
  Default=1
  Required=No

* **range1**:
  The range of the variogram in direction 1. Direction 1 is used for defining the
  anisotropy angle, [ **VGANGLE1**] , which is measured clockwise from the North. Note: It
  does not matter whether direction 1 is the major or minor axis.
  Range=Undefined
  Values=Undefined
  Default=1
  Required=No

* **range2**:
  The range of the variogram in direction 2, perpendicular to direction 1. (1)
  Range=Undefined
  Values=Undefined
  Default=1
  Required=No

* **vgangle1**:
  The angle between the N axis and direction 1, measured clockwise from N in degrees (0).
  Range=Undefined
  Values=Undefined
  Default=0
  Required=No

* **search**:
  Maximum search distance from centre of gravity of panel (9999).
  Range=Undefined
  Values=Undefined
  Default=9999
  Required=No

* **nmax**:
  The maximum number of samples to be used.
  Range=1,1900
  Values=Undefined
  Default=50
  Required=No

* **nsim**:
  The starting value for the number of simulated points in the X and Y directions for
  panel simulation. The default value of (20) means that the initial square grid has
  20x20=400 points.
  Range=Undefined
  Values=Undefined
  Default=20
  Required=No

* **minsim**:
  Minimum number of simulated points in panel. The default is (400).
  Range=Undefined
  Values=Undefined
  Default=400
  Required=No

* **vmethod**:
  Method for defining variogram parameters (1) Options: 1: Use parameters **NUGGET , VAR1
  , RANGE1 , RANGE2 ,** and **VGANGLE1**. This is consistent with earlier versions of
  **PANELK** and can be used for a single structure spherical model.; 2: Define parameters
  using interactive prompts. This method allows a selection of variogram models, defined
  using VGRAM.
  Range=1,2
  Values=1,2
  Default=1
  Required=No

* **vgram**:
  Variogram model type (1). Possible values are: Options: **1**: Single structure
  spherical model.; **2**: Two structure spherical model.; **3**: Linear model.; **4**: De
  Wijsian model.; **5**: Exponential model.; **6**: Gaussian model.; **7**: Experimental
  model.; **8**: NOT USED.; **9**: NOT USED.; **10**: Multi structure spherical model with
  anisotropy.
  Range=1,10
  Values=1,2,3,4,5,6,7,10
  Default=1
  Required=No

* **print**:
  Output control flag (1). Options: **0**: Minimum output. Just a summary of the sample
  data and a table of results is output.; **1**: As 0 plus a summary of all input files,
  fields and parameters.; **2**: As 1 plus a listing of all simulated points, plus all
  samples and kriged weights.; **3**: As 2 plus a listing of panel points.
  Range=0.3
  Values=0,1,2,3
  Default=1
  Required=No

In [ ]:
# Step 1: Connect to Datamine and Initialize Sandbox
import os
import shutil
import glob
import pandas as pd
from dmstudio import dmcommands, dmfiles, initialize, agent

# Connect to running Studio RM instance
dm_cmd = dmcommands.init(version='StudioRM')
dm_fil = dmfiles.init(version='StudioRM')
oScript = initialize.studio('StudioRM')
print(f"Connected to: {oScript.Caption}")

# Initialize active project sandbox using the shared test_sandbox project
notebook_folder = os.path.normpath(os.path.dirname(os.path.abspath('__file__'))).lower()
agent.initialize_sandbox(notebook_folder)

## Step 2: Introspect Schema
Always inspect the parameter schema for the process wrapper to see the expected input and output files, fields, and parameters.

In [ ]:
schema = agent.get_command_schema('panelk')
print(f"Process: {schema['name']}")
print("Parameters:")
for p in schema['parameters']:
    print(f"  - {p['name']}: default={p['default']}, annotation={p['annotation']}")

## Step 3: Prepare Inputs
We initialize the example project by copying the relevant standard datasets from the Datamine database locally to this sandbox folder using a `t_` prefix.

In [ ]:
# Step 3: Copy VBOP datasets dynamically from Database to test_sandbox
files_to_copy = [
    "_vb_assays.dmx",
    "_vb_collars.dmx",
    "_vb_surveys.dmx",
    "_vb_lithology.dmx",
    "_vb_epar.dmx",
    "_vb_spar.dmx",
    "_vb_vpar.dmx",
    "_vb_mod1.dmx",
    "_vb_SurfacePointsPt.dmx",
    "_vb_SurfaceTriangles.dmx"
]

agent.initialize_sandbox(notebook_folder, files_to_copy=files_to_copy)

## Step 4: Execute Process
Call the wrapper method with appropriate arguments. Below is the generated template execution call. Required parameters are active, and optional parameters are commented out.

In [ ]:
# Execute panelk
print("Running panelk...")
dm_cmd.panelk(
    inmods_i=['optional'],  # required
    out_o='t_panelk_out',  # required
    x_f='X',  # required
    y_f='Y',  # required
    value_f='optional',  # required
    panel_f='optional',  # required
    # nugget_p=0,  # optional
    # var1_p=1,  # optional
    # ranges_f=['optional'],  # optional
    # vgangle1_p=0,  # optional
    # search_p=9999,  # optional
    # nmax_p=50,  # optional
    # nsim_p=20,  # optional
    # minsim_p=400,  # optional
    # vmethod_p=1,  # optional
    # vgram_p=1,  # optional
    # print_p=1,  # optional
    # arguments='optional',  # optional
    # retrieval='optional',  # optional
)
print("panelk execution completed.")

## Step 5: Verify Results
Check that output files exist on disk and read them using pandas to verify the outputs.

In [ ]:
# Step 5: Verify results
output_file = 't_panelk_out.dmx'
if os.path.exists(output_file):
    df = agent.read_datamine(output_file)
    print(f"Output file loaded successfully. Rows: {len(df)}")
    print(df.head())
else:
    print("Output file not found (run Step 4 first)")

## Step 6: Clean up Project Folder
Always clean up generated temporary files to keep the directory clean.

In [ ]:
# Step 6: Clean up temporary files and generated artifacts
# 1. Clean up temporary files matching t_*.*
temp_files = glob.glob("t_*.*")
for f in temp_files:
    try:
        os.remove(f)
        print(f"Removed {os.path.basename(f)}")
    except Exception as e:
        print(f"Failed to remove {os.path.basename(f)}: {e}")

# 2. Clean up dynamic python initialization files (dmdir.py, __init__.py)
extra_files = ['dmdir.py', '__init__.py']
for f in extra_files:
    if os.path.exists(f):
        try:
            os.remove(f)
            print(f"Removed {f}")
        except Exception as e:
            print(f"Failed to remove {f}: {e}")

# 3. Clean up cache directories (__pycache__)
pycache = '__pycache__'
if os.path.exists(pycache):
    try:
        shutil.rmtree(pycache)
        print("Removed __pycache__")
    except Exception as e:
        print(f"Failed to remove __pycache__: {e}")